# 06 — Prompt Engineering and Structured Outputs

**Network LLM Engineering — Part II — Knowledge and Context**

### Learning goals
- Build a strong baseline before fine-tuning
- Use role, task, constraints, context and examples
- Validate structured output deterministically

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
).eval()

def render_chat(messages, generation=True):
    kw = dict(tokenize=False, add_generation_prompt=generation)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

@torch.inference_mode()
def generate(messages, max_new_tokens=160, temperature=0.0):
    text = render_chat(messages, True)
    toks = tokenizer(text, return_tensors="pt")
    dev = next(model.parameters()).device
    toks = {k:v.to(dev) for k,v in toks.items()}
    sample = temperature > 0
    out = model.generate(
        **toks, max_new_tokens=max_new_tokens,
        do_sample=sample,
        temperature=temperature if sample else None,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, toks["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## Prompt architecture

A robust operational prompt often has:

1. **Role/goal**
2. **Evidence/context**
3. **Task**
4. **Constraints**
5. **Output schema**
6. **Examples** when needed

Do not rely on vague phrases like "be accurate". Specify observable behavior.

In [ ]:
question = "Users can reach their default gateway but not 10.20.0.0/16."

weak = [{"role":"user","content":question}]
strong = [
    {"role":"system","content":
     "You are a careful network troubleshooter. Treat supplied observations as evidence, not proof of root cause. "
     "Give exactly three low-risk verification steps before recommending any change."},
    {"role":"user","content":question}
]

print("WEAK\n", generate(weak, 140))
print("\nSTRONG\n", generate(strong, 140))

## Structured output is a systems problem

Even if you ask for JSON, validate the result. A model can produce malformed data.
Production systems should use schemas, parsers, retry/fallback logic, and policy enforcement outside the LLM.

In [ ]:
from jsonschema import validate

schema = {
  "type":"object",
  "required":["diagnosis","next_checks","confidence"],
  "properties":{
    "diagnosis":{"type":"string"},
    "next_checks":{"type":"array","items":{"type":"string"}},
    "confidence":{"enum":["low","medium","high"]}
  }
}
example = {"diagnosis":"Return path or policy is a leading hypothesis",
           "next_checks":["Check route to 10.20.0.0/16","Check return route","Check ACL/firewall policy"],
           "confidence":"medium"}
validate(example, schema)
print("valid")

### Exercise

Create a prompt that forces the model to separate:
- observations,
- hypotheses,
- next checks,
- proposed change,
- confidence.

Then decide which fields should be model-generated and which should be validated by deterministic code.